# Toolbar

> View mode toggle, sort controls, and filter controls for the file browser.

In [ ]:
#| default_exp components.toolbar

In [ ]:
#| export
from typing import Any, Optional

from fasthtml.common import Div, Span, Button, Select, Option, Label

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_styles
from cjm_fasthtml_daisyui.components.data_input.select import select as dui_select, select_sizes
from cjm_fasthtml_daisyui.components.layout.join import join
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.borders import border, rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, justify, items, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_fasthtml_file_browser.core.config import FileBrowserConfig, ViewMode, FileColumn
from cjm_fasthtml_file_browser.core.models import BrowserState
from cjm_fasthtml_file_browser.components.item import BROWSER_ICONS

## View Mode Toggle

Buttons to switch between list and grid view.

In [ ]:
#| export
def render_view_toggle(
    current_mode: ViewMode,                 # Current view mode
    toggle_url: str,                        # URL for toggling view mode
    hx_target: Optional[str] = None,        # HTMX target for swaps
) -> Any:  # View toggle component
    """Render view mode toggle buttons (list/grid)."""
    def make_button(mode: ViewMode, icon_name: str, title: str):
        is_active = current_mode == mode
        btn_cls = combine_classes(
            btn, btn_sizes.sm,
            btn_colors.primary if is_active else btn_styles.ghost
        )
        
        attrs = {
            "cls": btn_cls,
            "title": title,
            "hx_post": toggle_url,
            "hx_vals": f'{{"view_mode": "{mode.value}"}}',
            "hx_swap": "outerHTML",
        }
        if hx_target:
            attrs["hx_target"] = hx_target
        
        icon_cls = str(text_dui.primary_content) if is_active else str(text_dui.base_content)
        return Button(
            lucide_icon(icon_name, size=4, cls=icon_cls),
            **attrs
        )
    
    return Div(
        make_button(ViewMode.LIST, BROWSER_ICONS["list_view"], "List view"),
        make_button(ViewMode.GRID, BROWSER_ICONS["grid_view"], "Grid view"),
        cls=combine_classes(join, rounded.lg)
    )

In [ ]:
from fasthtml.common import to_xml

# Test view toggle
toggle = render_view_toggle(ViewMode.LIST, "/toggle")
html = to_xml(toggle)
assert "<button" in html.lower()
assert "btn-primary" in html  # Active button has primary color
assert "list" in html.lower() or "<svg" in html.lower()  # List icon

# Test grid active
toggle_grid = render_view_toggle(ViewMode.GRID, "/toggle")
html = to_xml(toggle_grid)
assert "<button" in html.lower()

print("View toggle tests passed!")

View toggle tests passed!


## Sort Controls

Dropdown and direction toggle for sorting files.

In [ ]:
#| export
def render_sort_controls(
    current_sort: str,                      # Current sort field
    descending: bool,                       # Current sort direction
    config: FileBrowserConfig,              # Browser configuration
    sort_url: str,                          # URL for changing sort
    hx_target: Optional[str] = None,        # HTMX target for swaps
) -> Any:  # Sort controls component
    """Render sort dropdown and direction toggle."""
    # Sort options based on configured columns
    sort_options = [
        ("name", "Name"),
        ("size", "Size"),
        ("modified", "Modified"),
        ("type", "Type"),
    ]
    
    # Build select options
    options = []
    for value, label in sort_options:
        options.append(Option(
            label,
            value=value,
            selected=(value == current_sort)
        ))
    
    # Sort select
    select_attrs = {
        "name": "sort_by",
        "cls": combine_classes(dui_select, select_sizes.sm, w(32)),
        "hx_post": sort_url,
        "hx_swap": "outerHTML",
        "hx_trigger": "change",
        "hx_include": "[name='sort_descending']",  # Include direction
    }
    if hx_target:
        select_attrs["hx_target"] = hx_target
    
    sort_select = Select(*options, **select_attrs)
    
    # Direction toggle button
    icon_name = BROWSER_ICONS["sort_desc"] if descending else BROWSER_ICONS["sort_asc"]
    dir_title = "Sort descending" if descending else "Sort ascending"
    
    dir_attrs = {
        "cls": combine_classes(btn, btn_styles.ghost, btn_sizes.sm),
        "title": dir_title,
        "hx_post": sort_url,
        "hx_vals": f'{{"sort_by": "{current_sort}", "sort_descending": "{str(not descending).lower()}"}}',
        "hx_swap": "outerHTML",
    }
    if hx_target:
        dir_attrs["hx_target"] = hx_target
    
    dir_button = Button(
        lucide_icon(icon_name, size=4),
        **dir_attrs
    )
    
    # Hidden input for current direction (for select hx_include)
    hidden_dir = Div(
        f'<input type="hidden" name="sort_descending" value="{str(descending).lower()}" />',
        style="display:none"
    )
    
    return Div(
        Label("Sort:", cls=combine_classes(font_size.sm, text_dui.base_content.opacity(70), m.r(2))),
        sort_select,
        dir_button,
        hidden_dir,
        cls=combine_classes(flex_display, items.center, gap(1))
    )

In [ ]:
# Test sort controls
config = FileBrowserConfig()
sort_ctrl = render_sort_controls("name", False, config, "/sort")
html = to_xml(sort_ctrl)
assert "<select" in html.lower()
assert "Name" in html
assert "Size" in html
assert "<button" in html.lower()  # Direction toggle

print("Sort controls tests passed!")

Sort controls tests passed!


## Complete Toolbar

Full toolbar combining view toggle, sort controls, and optional filter info.

In [ ]:
#| export
def render_toolbar(
    state: BrowserState,                    # Current browser state
    config: FileBrowserConfig,              # Browser configuration
    toggle_url: str,                        # URL for view toggle
    sort_url: str,                          # URL for sort changes
    hx_target: Optional[str] = None,        # HTMX target for swaps
    toolbar_id: Optional[str] = None,       # HTML ID for toolbar
) -> Any:  # Toolbar component
    """Render the complete toolbar."""
    children = []
    
    # View toggle (if allowed)
    if config.view.allow_mode_toggle:
        children.append(render_view_toggle(
            ViewMode(state.view_mode),
            toggle_url,
            hx_target
        ))
    
    # Spacer
    children.append(Div(cls=str(grow())))
    
    # Sort controls (if allowed)
    if config.view.allow_sort_toggle:
        children.append(render_sort_controls(
            state.sort_by,
            state.sort_descending,
            config,
            sort_url,
            hx_target
        ))
    
    # Active filter indicator (if extensions filtered)
    if state.filter_extensions:
        ext_list = ", ".join(state.filter_extensions)
        children.append(Div(
            Span(f"Filtered: {ext_list}", cls=combine_classes(
                font_size.xs, text_dui.base_content.opacity(60)
            )),
            cls=str(m.l(4))
        ))
    
    # Build toolbar
    toolbar_attrs = {
        "cls": combine_classes(
            flex_display, items.center, gap(2),
            p.x(2), p.y(1),
            border_dui.base_300, border.b()
        )
    }
    if toolbar_id:
        toolbar_attrs["id"] = toolbar_id
    
    return Div(*children, **toolbar_attrs)

In [ ]:
# Test complete toolbar
config = FileBrowserConfig()
state = BrowserState(current_path="/home", view_mode="list", sort_by="name")

toolbar = render_toolbar(state, config, "/toggle", "/sort")
html = to_xml(toolbar)
assert "<button" in html.lower()  # View toggle buttons
assert "<select" in html.lower()  # Sort select

# Test with filter active
state_filtered = BrowserState(
    current_path="/home",
    filter_extensions=[".py", ".txt"]
)
toolbar_filtered = render_toolbar(state_filtered, config, "/toggle", "/sort")
html = to_xml(toolbar_filtered)
assert "Filtered" in html

print("Complete toolbar tests passed!")

Complete toolbar tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()